# 06회차 실습 노트북 — 데이터 분석 실습 — KNN·SVM·분할·표준화

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/06_knn_svm.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/06_knn_svm.html](https://adminmrobo.github.io/fintech_data_analysis/06_knn_svm.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

## 1. 순서대로 자르면 생기는 일
이탈 여부로 정렬된 데이터의 앞 75%를 학습, 뒤 25%를 시험으로 쓰면?

In [2]:
b = load('Revolution/Bank_customers_data.csv')
b['y'] = (b.Attrition_Flag == 'Attrited Customer').astype(int)   # 1 = 이탈 고객
print(b.shape, '이탈 비율:', round(b.y.mean(), 3))
bs = b.sort_values('y'); cut = int(len(bs) * .75)
print('학습 세트 이탈 고객:', bs.y.values[:cut].sum(), ' / 시험 세트 이탈 고객:', bs.y.values[cut:].sum(), '명 중', len(bs) - cut)

(10127, 22) 이탈 비율: 0.161
학습 세트 이탈 고객: 0  / 시험 세트 이탈 고객: 1627 명 중 2532


## 2. 대결 재현: 표준화한 SVM vs 그대로 SVM
거래 횟수(10~139)와 거래 금액(510~18,484)은 단위 크기가 100배 넘게 다릅니다.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import recall_score
X = b[['Total_Trans_Ct', 'Total_Trans_Amt']].values.astype(float)
Xtr, Xte, ytr, yte = train_test_split(X, b.y.values, test_size=.25, random_state=RS, stratify=b.y)
sc = StandardScaler().fit(Xtr)
raw = SVC(kernel='rbf', C=1).fit(Xtr, ytr)
std = SVC(kernel='rbf', C=1).fit(sc.transform(Xtr), ytr)
print('그대로   정확도', round(raw.score(Xte, yte), 4), ' 이탈 재현율', round(recall_score(yte, raw.predict(Xte)), 4))
print('표준화   정확도', round(std.score(sc.transform(Xte), yte), 4), ' 이탈 재현율', round(recall_score(yte, std.predict(sc.transform(Xte))), 4))
print('시험 세트 이탈 고객 수:', yte.sum())

그대로   정확도 0.8393  이탈 재현율 0.0


표준화   정확도 0.887  이탈 재현율 0.4152
시험 세트 이탈 고객 수: 407


**AI 판단 연습** — "그대로" 모델의 정확도 83.9%는 무엇과 같은 숫자인가요? (5회차의 "모두 이탈 안 함" 기준과 비교)